# **Prova Estrazione Highlights**

In questa prova utilizzo il dataset Learning From The Worst la cui definizione usata è:

"‘Hate’ is defined as “abusive speech targeting specific group characteristics, such as ethnic origin, religion, gender, or sexual orientation."

Le componenti sono Abuse, Ethnicity, Gender, Religion, Sexual Orientation.

Questa prova è con Mistral-7b-Instruct-v0.3

# HF login e imports

In [ ]:
!pip install -U huggingface_hub

In [ ]:
!hf auth login --token hf_**************

In [ ]:
!pip install --upgrade transformers accelerate bitsandbytes

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import numpy as np
import json
import random
from IPython.display import display, HTML

# Data Processing

In [ ]:
!gdown 1ygTn8Q6xVk_RNTae-0W-SigKdjHOkUTx

In [ ]:
df=pd.read_csv('/content/sampled_rows_LFTW.csv')
df.head()

In [ ]:
texts=df['text']
texts

# Load Model

In [ ]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_name="mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quantization_config
  )

# Prompt

In [ ]:
prompt = [
    {
        "role": "system",
        "content": "You are a data annotator specialized in hate speech detection."
    },
    {
        "role": "user",
        "content": """Read the provided TEXT and extract the exact substring that correspond to the following Hate Speech components.
        If a component is not present in the text, you must return the string "absent".

        Below you find the components to identify in the given text:

        - Abuse:  Content which involves improper treatment of a person or group, often to unfairly or improperly gain benefit.
        - Derogation: Content which explicitly attacks, demonizes, demeans or insults a group
        - Animosity: Content which expresses abuse against a group in an implicit or subtle manner.
        - Threatening language: Content which expresses intention to, support for, or encourages inflicting harm on a group, or identified members of the group.
        - Support for hateful entities: Content which explicitly glorifies, justifies or supports hateful actions, events, organizations, tropes and individuals.
        - Dehumanization Content which ‘perceiv[es] or treat[s] people as less than human’
        - Ethnicity: Any reference to a person's or group's ethnic origin.
        - Gender: Any reference to a person's or group's gender identity or biological sex.
        - Religion: Any reference to a person's or group's religious beliefs, practices, or affiliation.
        - Sexual Orientation: Any reference to a person's or group's sexual orientation.

        OUTPUT FORMAT:
        {
            "Abuse": "extracted substring or absent",
            "Derogation": "extracted substring or absent",
            "Animosity": "extracted substring or absent",
            "Threatening language": "extracted substring or absent",
            "Support for hateful entities": "extracted substring or absent",
            "Dehumanization": "extracted substring or absent",
            "Ethnicity": "extracted substring or absent",
            "Gender": "extracted substring or absent",
            "Religion": "extracted substring or absent",
            "Sexual Orientation": "extracted substring or absent"
        }

        RULES:
        1- You must extract exact quotes from the text. Do not paraphrase or change any words.
        2- Do not output any conversational text, greetings, introductions, or explanations.
        3- Your response must be ONLY the raw json object. Do not wrap the json in markdown formatting.

        TEXT: "{text}"

        ANSWER:

        """
    }
]

In [ ]:
def prepare_prompts(texts, prompt_template, tokenizer):
  """
    This function format input text samples into instructions prompts.

    Inputs:
      texts: input texts to classify via prompting
      prompt_template: the prompt template provided in this assignment
      tokenizer: the transformers Tokenizer object instance associated
      with the chosen model card

    Outputs:
      input texts to classify in the form of instruction prompts
  """
  formatted_prompts=[]

  for text in texts:
    prompt_copy = [
            {"role": prompt_template[0]["role"], "content": prompt_template[0]["content"]},
            {
                "role": prompt_template[1]["role"],
                "content": prompt_template[1]["content"].replace("{text}", text)
            }
        ]


    formatted_prompt = tokenizer.apply_chat_template(
      prompt_copy,
      add_generation_prompt=True,
      tokenize=True,
      return_dict=True,
      return_tensors="pt",
    )

    formatted_prompts.append(formatted_prompt)


  return formatted_prompts

# Inference

In [ ]:
def generate_responses(model, prompt_examples, tokenizer, verbose=False):
  """
    This function implements the inference loop for a LLM model.
    Given a set of examples, the model is tasked to generate
    a response.

    Inputs:
      model: LLM model instance for prompting
      prompt_examples: pre-processed text samples

    Outputs:
      generated responses
  """
  outputs=[]
  for index, prompt_example in enumerate(prompt_examples):
    if verbose and index % 10 == 0:
      print(f"generating response n.{index}...")
    prompt_example=prompt_example.to(model.device)
    output = model.generate(**prompt_example, max_new_tokens=500,pad_token_id=tokenizer.eos_token_id)
    output = tokenizer.decode(output[0][prompt_example["input_ids"].shape[-1]:], skip_special_tokens=True)
    outputs.append(output)

  return outputs

In [ ]:
outputs = generate_responses(model, prepare_prompts(texts, prompt, tokenizer), tokenizer, verbose=True)
#32

In [ ]:
parsed_outputs = []
json_errors = []
for output in outputs:
    try:
        parsed_outputs.append(json.loads(output))
    except json.JSONDecodeError:
        json_errors.append(output)

with open("results_LFTW_3.json", "w", encoding="utf-8") as f:
    json.dump(parsed_outputs, f, ensure_ascii=False, indent=2)

with open("json_errors_LFTW_3.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(json_errors))
    f.write("\n")
    f.write("--"*20)

# Extraction Results

In [ ]:
texts = texts.reset_index(drop=True)

In [ ]:
for i in range(len(outputs)):
  print(texts[i])
  print(outputs[i])
  print('-'*20)

In [ ]:
def highlights(texts, outputs):

  """
      This function displays the given text with the model extracted highlights.

  """

  colori = {
        "Abuse": "#FFB3B3",
        "Ethnicity": "#B3D9FF",
        "Gender": "#E6B3FF",
        "Religion": "#B3FFB3",
        "Sexual Orientation": "#FFD9B3"
      }

  testo_colorato = texts

  for categoria, estratto in outputs.items():
      if estratto and estratto.lower() != "null" and estratto in testo_colorato:
          colore = colori.get(categoria, "#DDDDDD")
          tag_html = f"<mark style='background-color: {colore}; padding: 2px 4px; border-radius: 4px;'>{estratto} <span style='font-size: 0.7em; font-weight: bold;'>[{categoria}]</span></mark>"
          testo_colorato = testo_colorato.replace(estratto, tag_html)

  display(HTML(f"<p style='line-height: 1.8; font-size: 16px;'>{testo_colorato}</p>"))


In [ ]:
for i in range(len(outputs)):
    parsed_output = {}
    try:
        parsed_output = json.loads(outputs[i])
    except json.JSONDecodeError as e:
        print(f"JSONDecodeError: {e} for text: {texts[i]}, model output: {outputs[i]}")

    highlights(texts[i], parsed_output)